# Fase 1 — Grid Search del sample para modelo de gustos

Evalúa 48 combinaciones de filtros (`window_days`, `min_logins`, `min_lifespan`)
sobre `users.csv`. Calcula cobertura de los 11 CSVs del proyecto y selecciona la
configuración óptima.

NO genera parquets de features. NO entrena modelos. Solo evalúa filtros.

In [1]:
# [SETUP] Imports y paths
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from itertools import product
import time, re

ROOT = Path.cwd().parents[1] if Path.cwd().name == 'fase4_gustos' else Path.cwd()
DATA_RAW = ROOT / 'data' / 'data_raw'
DATA_QC = ROOT / 'data' / 'data_qc'
INFORMES = ROOT / 'informes' / 'fase4_gustos' / '01_grid_search'
INFORMES.mkdir(parents=True, exist_ok=True)

REFERENCE_DATE = pd.Timestamp('2026-04-04', tz='UTC')
print(f'Output dir: {INFORMES}')
print(f'REFERENCE_DATE: {REFERENCE_DATE}')

OID_RE = re.compile(r'[0-9a-f]{24}')
def clean_id(s):
    if pd.isna(s): return None
    m = OID_RE.search(str(s))
    return m.group(0) if m else None


Output dir: /Users/jezquerro/Documents/tfg/informes/fase4_gustos/01_grid_search
REFERENCE_DATE: 2026-04-04 00:00:00+00:00


In [2]:
# [EXEC] Cargar users.csv UNA SOLA VEZ
t0 = time.time()
users_raw = pd.read_csv(
    DATA_RAW / 'users.csv',
    usecols=['_id', 'last_login_date', 'num_logins', 'tutorial_done', 'created_at'],
    low_memory=False,
)
users_raw['user_id'] = users_raw['_id'].apply(clean_id)
users_raw['last_login_dt'] = pd.to_datetime(users_raw['last_login_date'], unit='s', utc=True, errors='coerce')
users_raw['created_at_dt'] = pd.to_datetime(users_raw['created_at'], errors='coerce', utc=True)
users_raw = users_raw.dropna(subset=['user_id', 'last_login_dt', 'created_at_dt'])
print(f'users_raw: {len(users_raw):,} filas validas en {time.time()-t0:.1f}s')
print(f'  rango last_login: {users_raw["last_login_dt"].min()} -> {users_raw["last_login_dt"].max()}')


users_raw: 1,164,298 filas validas en 2.6s
  rango last_login: 2020-04-30 08:42:41+00:00 -> 2026-04-04 18:23:37+00:00


In [3]:
# [EXEC] Construir mapping char->user y pre-calcular sets de user_id por CSV
t0 = time.time()

chars = pd.read_csv(DATA_RAW / 'characters.csv', usecols=['_id', 'user_id'], low_memory=False)
chars['char_id'] = chars['_id'].apply(clean_id)
chars['user_id'] = chars['user_id'].apply(clean_id)
chars = chars.dropna(subset=['char_id', 'user_id'])
char_to_user = dict(zip(chars['char_id'], chars['user_id']))
users_with_char = set(chars['user_id'])
print(f'characters: {len(chars):,} filas | char_to_user: {len(char_to_user):,} | usuarios: {len(users_with_char):,}')

user_sets = {'characters': users_with_char}

u_items = set()
for chunk in pd.read_csv(DATA_RAW / 'user_items.csv', usecols=['user_id'], chunksize=500_000, low_memory=False):
    u_items.update(chunk['user_id'].dropna().apply(clean_id))
user_sets['items'] = u_items
print(f'items: {len(u_items):,} usuarios')

coll = pd.read_csv(DATA_RAW / 'user_items_collection.csv', usecols=['user_id'], low_memory=False)
user_sets['collection'] = set(coll['user_id'].dropna().apply(clean_id))
print(f'collection: {len(user_sets["collection"]):,} usuarios')

dev = pd.read_csv(DATA_RAW / 'devices.csv', usecols=['user_id'], low_memory=False)
user_sets['devices'] = set(dev['user_id'].dropna().apply(clean_id))
print(f'devices: {len(user_sets["devices"]):,} usuarios')

dr = pd.read_csv(DATA_RAW / 'user_daily_rewards.csv', usecols=['user_id'], low_memory=False)
user_sets['daily_rewards'] = set(dr['user_id'].dropna().apply(clean_id))
print(f'daily_rewards: {len(user_sets["daily_rewards"]):,} usuarios')

u_fights = set()
for chunk in pd.read_csv(DATA_RAW / 'fights_log.csv', usecols=['player_id'], chunksize=500_000, low_memory=False):
    chunk_chars = chunk['player_id'].dropna().apply(clean_id)
    chunk_users = chunk_chars.map(char_to_user).dropna()
    u_fights.update(chunk_users)
user_sets['fights'] = u_fights
print(f'fights: {len(u_fights):,} usuarios')

u_curr = set()
for chunk in pd.read_csv(DATA_RAW / 'currency_transactions.csv', usecols=['user_id'], chunksize=500_000, low_memory=False):
    u_curr.update(chunk['user_id'].dropna().apply(clean_id))
user_sets['currency'] = u_curr
print(f'currency: {len(u_curr):,} usuarios')

ar = pd.read_csv(DATA_RAW / 'arena_log.csv', usecols=['attacker_id'], low_memory=False)
ar_chars = ar['attacker_id'].dropna().apply(clean_id)
ar_users = ar_chars.map(char_to_user).dropna()
user_sets['arena'] = set(ar_users)
print(f'arena: {len(user_sets["arena"]):,} usuarios')

iap_c = pd.read_csv(DATA_RAW / 'processed_consumables_iaps.csv', usecols=['user_id'], low_memory=False)
iap_s = pd.read_csv(DATA_RAW / 'processed_subscriptions_iaps.csv', usecols=['user_id'], low_memory=False)
u_iap = set(iap_c['user_id'].dropna().apply(clean_id)) | set(iap_s['user_id'].dropna().apply(clean_id))
user_sets['iaps'] = u_iap
print(f'iaps: {len(u_iap):,} usuarios')

fb = pd.read_csv(DATA_RAW / 'support_user_feedback_by_type.csv', usecols=['user_id'], low_memory=False)
user_sets['feedback'] = set(fb['user_id'].dropna().apply(clean_id))
print(f'feedback: {len(user_sets["feedback"]):,} usuarios')

print(f'\nuser_sets totales construidos en {time.time()-t0:.1f}s')


characters: 1,336,143 filas | char_to_user: 1,336,143 | usuarios: 1,164,543


items: 912,756 usuarios


collection: 657,168 usuarios


devices: 1,076,057 usuarios


daily_rewards: 705,795 usuarios


fights: 68,032 usuarios


currency: 67,085 usuarios


arena: 8,284 usuarios


iaps: 31,250 usuarios
feedback: 8,750 usuarios

user_sets totales construidos en 77.4s


In [4]:
# [EXEC] Cargar sample de churn para overlap
churn_sample_df = pd.read_parquet(DATA_QC / 'sample_user_ids_cutoff.parquet')
churn_sample_df['user_id'] = churn_sample_df['user_id'].apply(clean_id)
churn_sample_set = set(churn_sample_df['user_id'].dropna())
print(f'Sample de churn: {len(churn_sample_set):,}')


Sample de churn: 25,200


In [5]:
# [EXEC] Funcion reutilizable para evaluar una combinacion
def evaluate_config(users_raw, window_days, min_logins, min_lifespan,
                    user_sets, churn_sample_set, fixed_min_year=2018):
    df = users_raw
    if 'tutorial_done' in df.columns:
        df = df[df['tutorial_done'] == True]
    df = df[df['last_login_dt'].dt.year > 2018]
    df = df[df['created_at_dt'].dt.year >= fixed_min_year]
    cutoff_active = REFERENCE_DATE - pd.Timedelta(days=window_days)
    df = df[df['last_login_dt'] >= cutoff_active]
    df = df[df['num_logins'] >= min_logins]
    df = df.assign(account_age_days=(REFERENCE_DATE - df['created_at_dt']).dt.days)
    df = df[df['account_age_days'] >= min_lifespan]

    n_sample = len(df)
    sample_users = set(df['user_id'].dropna())

    coverages = {}
    for csv_name, csv_users in user_sets.items():
        coverages[f'coverage_{csv_name}'] = (
            len(sample_users & csv_users) / n_sample if n_sample > 0 else 0.0
        )

    overlap_n = len(sample_users & churn_sample_set)
    pct_overlap = overlap_n / len(churn_sample_set) if churn_sample_set else 0

    cov_tier1 = float(np.mean([
        coverages['coverage_characters'], coverages['coverage_items'],
        coverages['coverage_collection'], coverages['coverage_devices'],
        coverages['coverage_daily_rewards'],
    ]))
    cov_tier12 = 0.7 * cov_tier1 + 0.3 * float(np.mean([
        coverages['coverage_fights'], coverages['coverage_currency'],
    ]))
    mean_logins = float(df['num_logins'].mean()) if n_sample else 0.0
    viable = (
        80_000 <= n_sample <= 200_000 and
        cov_tier1 >= 0.90 and
        cov_tier12 >= 0.55 and
        mean_logins >= 10
    )
    return {
        'window_days': window_days,
        'min_logins': min_logins,
        'min_lifespan': min_lifespan,
        'n_sample': int(n_sample),
        'pct_retained': round(n_sample / len(users_raw) * 100, 2),
        **{k: round(v, 4) for k, v in coverages.items()},
        'coverage_tier1_score': round(cov_tier1, 4),
        'coverage_tier12_score': round(cov_tier12, 4),
        'mean_account_age_days': round(float(df['account_age_days'].mean()) if n_sample else 0, 1),
        'median_account_age_days': float(df['account_age_days'].median()) if n_sample else 0,
        'mean_num_logins': round(mean_logins, 1),
        'pct_overlap_with_churn_sample': round(pct_overlap, 4),
        'overlap_n': int(overlap_n),
        'viable': bool(viable),
        'score_main': round(cov_tier12 * np.log10(max(n_sample, 1)), 4),
    }
print('evaluate_config definida')


evaluate_config definida


In [6]:
# [EXEC] Ejecutar grid 4x4x3 = 48 combinaciones
combinations = list(product([30, 45, 60, 90], [2, 5, 10, 20], [0, 7, 30]))
print(f'Ejecutando {len(combinations)} combinaciones...')
results = []
t0 = time.time()
for i, (w, l, lf) in enumerate(combinations, 1):
    res = evaluate_config(users_raw, w, l, lf, user_sets, churn_sample_set)
    results.append(res)
    if i % 8 == 0:
        print(f'  [{i:2d}/{len(combinations)}] w={w:>2} l={l:>2} lf={lf:>2} -> '
              f'N={res["n_sample"]:>7,} cov_t12={res["coverage_tier12_score"]:.3f} viable={res["viable"]}')
print(f'\nGrid completo en {time.time()-t0:.1f}s')
results_df = pd.DataFrame(results)
results_df.to_csv(INFORMES / 'grid_search_results.csv', index=False)
print(f'OK grid_search_results.csv guardado ({len(results_df)} filas)')


Ejecutando 48 combinaciones...


  [ 8/48] w=30 l=10 lf= 7 -> N= 11,640 cov_t12=0.953 viable=False


  [16/48] w=45 l= 5 lf= 0 -> N= 33,742 cov_t12=0.883 viable=False


  [24/48] w=45 l=20 lf=30 -> N=  6,995 cov_t12=0.853 viable=False


  [32/48] w=60 l=10 lf= 7 -> N= 21,654 cov_t12=0.834 viable=False


  [40/48] w=90 l= 5 lf= 0 -> N= 63,433 cov_t12=0.795 viable=False


  [48/48] w=90 l=20 lf=30 -> N= 13,446 cov_t12=0.776 viable=False

Grid completo en 5.5s
OK grid_search_results.csv guardado (48 filas)


In [7]:
# [ANALYSIS] Filtrar viables y rankear
cols_show = ['window_days', 'min_logins', 'min_lifespan', 'n_sample',
             'coverage_tier1_score', 'coverage_tier12_score',
             'mean_num_logins', 'pct_overlap_with_churn_sample', 'score_main']

viable = results_df[results_df['viable']].sort_values('score_main', ascending=False)
print(f'Configuraciones viables: {len(viable)}/{len(results_df)}')

if len(viable) > 0:
    viable.to_csv(INFORMES / 'viable_configs.csv', index=False)
    print('\nTOP 5 viables:')
    print(viable.head(5)[cols_show].to_string(index=False))
else:
    print('\nNO HAY CONFIGURACIONES VIABLES')

top5 = results_df.sort_values('score_main', ascending=False).head(5)
top5.to_csv(INFORMES / 'top_configs.csv', index=False)
print('\nTOP 5 general (sin filtro de viabilidad):')
print(top5[cols_show].to_string(index=False))


Configuraciones viables: 4/48

TOP 5 viables:
 window_days  min_logins  min_lifespan  n_sample  coverage_tier1_score  coverage_tier12_score  mean_num_logins  pct_overlap_with_churn_sample  score_main
          60           2             0     86228                0.9941                 0.8387             50.2                         0.3471      4.1395
          90           2             0    127821                0.9938                 0.7920             39.8                         0.5326      4.0445
          90           2             7    121129                0.9938                 0.7807             41.7                         0.5326      3.9687
          90           2            30     96338                0.9937                 0.7254             50.3                         0.5326      3.6154

TOP 5 general (sin filtro de viabilidad):
 window_days  min_logins  min_lifespan  n_sample  coverage_tier1_score  coverage_tier12_score  mean_num_logins  pct_overlap_with_churn_sample

In [8]:
# [ANALYSIS] Heatmaps (agrupando min_lifespan por mediana)
agg = results_df.groupby(['window_days', 'min_logins']).agg(
    n_sample=('n_sample', 'median'),
    coverage_tier12_score=('coverage_tier12_score', 'median'),
    coverage_tier1_score=('coverage_tier1_score', 'median'),
).reset_index()

for metric, fname, cmap, fmt in [
    ('coverage_tier12_score', 'coverage_heatmap.png', 'RdYlGn', '.3f'),
    ('n_sample', 'n_sample_heatmap.png', 'viridis', ',.0f'),
]:
    pivot = agg.pivot(index='window_days', columns='min_logins', values=metric)
    fig, ax = plt.subplots(figsize=(8, 5))
    im = ax.imshow(pivot.values, cmap=cmap, aspect='auto')
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_xlabel('min_num_logins')
    ax.set_ylabel('window_days_active')
    ax.set_title(f'{metric} (mediana sobre min_lifespan)')
    for r in range(len(pivot.index)):
        for c in range(len(pivot.columns)):
            v = pivot.iloc[r, c]
            txt = format(v, fmt)
            ax.text(c, r, txt, ha='center', va='center', color='black', fontsize=9)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.savefig(INFORMES / fname, dpi=120, bbox_inches='tight')
    plt.close()
    print(f'OK {fname} guardado')


OK coverage_heatmap.png guardado


OK n_sample_heatmap.png guardado


In [9]:
# [REPORT] recommendation.md
if len(viable) > 0:
    best = viable.iloc[0]
    status = 'VIABLE'
    note_estado = 'Configuracion elegida cumple los 4 criterios de viabilidad.'
else:
    best = results_df.sort_values('score_main', ascending=False).iloc[0]
    status = 'MEJOR DENTRO DE LO INVIABLE'
    note_estado = 'Ninguna configuracion cumple los 4 criterios de viabilidad simultaneamente.'

alt_block = (viable.head(5)[cols_show].to_string(index=False)
             if len(viable) > 0 else 'Sin viables - ver top_configs.csv para alternativas.')

md = f"""# Grid Search - Recomendacion de configuracion para sample de gustos

**Fecha**: {time.strftime('%Y-%m-%d %H:%M')}
**Combinaciones evaluadas**: {len(results_df)}
**Configuraciones viables**: {len(viable)}

## Configuracion recomendada - {status}

```python
WINDOW_DAYS_ACTIVE = {int(best["window_days"])}
MIN_NUM_LOGINS     = {int(best["min_logins"])}
MIN_LIFESPAN_DAYS  = {int(best["min_lifespan"])}
```

{note_estado}

### Metricas resultantes

| Metrica | Valor |
|---|---:|
| N_SAMPLE | {int(best["n_sample"]):,} |
| Pct retenido sobre users.csv | {best["pct_retained"]}% |
| Cobertura Tier 1 (estables) | {best["coverage_tier1_score"]:.2%} |
| Cobertura Tier 1+2 (incluye fights/currency) | {best["coverage_tier12_score"]:.2%} |
| Edad media de cuenta (dias) | {best["mean_account_age_days"]:.0f} |
| Edad mediana de cuenta (dias) | {best["median_account_age_days"]:.0f} |
| Logins medios lifetime | {best["mean_num_logins"]:.1f} |
| Overlap con sample de churn | {best["pct_overlap_with_churn_sample"]:.2%} ({int(best["overlap_n"]):,} usuarios) |
| Score principal | {best["score_main"]:.4f} |

### Coberturas detalladas por CSV

| CSV | Tier | Cobertura |
|---|---|---:|
| characters | 1 | {best["coverage_characters"]:.2%} |
| items | 1 | {best["coverage_items"]:.2%} |
| collection | 1 | {best["coverage_collection"]:.2%} |
| devices | 1 | {best["coverage_devices"]:.2%} |
| daily_rewards | 1 | {best["coverage_daily_rewards"]:.2%} |
| fights (30d) | 2 | {best["coverage_fights"]:.2%} |
| currency (30d) | 2 | {best["coverage_currency"]:.2%} |
| arena (30d) | 3 | {best["coverage_arena"]:.2%} |
| iaps (lifetime) | 3 | {best["coverage_iaps"]:.2%} |
| feedback (lifetime) | 3 | {best["coverage_feedback"]:.2%} |

## Top 5 viables

```
{alt_block}
```

## Outputs

| Archivo | Contenido |
|---|---|
| `grid_search_results.csv` | Las {len(results_df)} combinaciones |
| `top_configs.csv` | Top 5 ranking por score_main |
| `viable_configs.csv` | Subset que pasa filtros de viabilidad |
| `coverage_heatmap.png` | Heatmap window x min_logins (cobertura Tier 1+2) |
| `n_sample_heatmap.png` | Heatmap window x min_logins (N_SAMPLE) |

## Proximo paso

Pasar a Fase 2 (feature engineering especifica para gustos) aplicando la configuracion
recomendada al sample de jugadores activos.
"""

(INFORMES / 'recommendation.md').write_text(md)
print(md)


# Grid Search - Recomendacion de configuracion para sample de gustos

**Fecha**: 2026-05-10 17:17
**Combinaciones evaluadas**: 48
**Configuraciones viables**: 4

## Configuracion recomendada - VIABLE

```python
WINDOW_DAYS_ACTIVE = 60
MIN_NUM_LOGINS     = 2
MIN_LIFESPAN_DAYS  = 0
```

Configuracion elegida cumple los 4 criterios de viabilidad.

### Metricas resultantes

| Metrica | Valor |
|---|---:|
| N_SAMPLE | 86,228 |
| Pct retenido sobre users.csv | 7.41% |
| Cobertura Tier 1 (estables) | 99.41% |
| Cobertura Tier 1+2 (incluye fights/currency) | 83.87% |
| Edad media de cuenta (dias) | 79 |
| Edad mediana de cuenta (dias) | 40 |
| Logins medios lifetime | 50.2 |
| Overlap con sample de churn | 34.71% (8,748 usuarios) |
| Score principal | 4.1395 |

### Coberturas detalladas por CSV

| CSV | Tier | Cobertura |
|---|---|---:|
| characters | 1 | 100.00% |
| items | 1 | 100.00% |
| collection | 1 | 100.00% |
| devices | 1 | 97.27% |
| daily_rewards | 1 | 99.77% |
| fights (30d) | 2 | 